In [0]:
from delta.tables import DeltaTable

DeltaTable.createIfNotExists(spark)\
    .tableName('demo')\
        .addColumn('id','long') \
            .addColumn('name','string')\
                .addColumn('amount','double')\
                .execute()

In [0]:
df = spark.createDataFrame([(1,'a',100.00),(2,'b',200.00),(3,'c',300.00),(4,'d',400.00),(5,'e',500.00)],['id','name','amount'])
df.printSchema()
df.write.format('delta').mode('overwrite').saveAsTable('demo')

root
 |-- id: long (nullable = true)
 |-- name: string (nullable = true)
 |-- amount: double (nullable = true)



In [0]:
df1 = spark.read.table('demo')
display(df1)

id,name,amount
1,a,100.0
2,b,200.0
3,c,300.0
4,d,400.0
5,e,500.0


In [0]:
%sql
describe history demo

version,timestamp,userId,userName,operation,operationParameters,job,notebook,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
1,2026-08-27T17:16:24Z,147836707444603,anooptu@gmail.com,CREATE OR REPLACE TABLE AS SELECT,"Map(partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, properties -> {""delta.parquet.compression.codec"":""zstd"",""delta.enableDeletionVectors"":""true""}, statsOnLoad -> false)",null,List(3175228607985161),0826-063441-ydade36q,0,WriteSerializable,false,"Map(numFiles -> 1, numRemovedFiles -> 0, numRemovedBytes -> 0, numOutputRows -> 5, numOutputBytes -> 1393)",null,Databricks-Runtime/16.4.x-scala2.13
0,2026-08-27T17:15:40Z,147836707444603,anooptu@gmail.com,CREATE TABLE,"Map(partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, properties -> {""delta.parquet.compression.codec"":""zstd"",""delta.enableDeletionVectors"":""true""}, statsOnLoad -> false)",null,List(3175228607985161),0826-063441-ydade36q,null,WriteSerializable,true,Map(),null,Databricks-Runtime/16.4.x-scala2.13


In [0]:
%sql
alter table demo
set tblproperties ('delta.enableChangeDataFeed' = 'true')

In [0]:
%sql
describe history demo

version,timestamp,userId,userName,operation,operationParameters,job,notebook,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
2,2026-08-27T17:18:09Z,147836707444603,anooptu@gmail.com,SET TBLPROPERTIES,"Map(properties -> {""delta.enableChangeDataFeed"":""true""})",null,List(3175228607985161),0826-063441-ydade36q,1,WriteSerializable,true,Map(),null,Databricks-Runtime/16.4.x-scala2.13
1,2026-08-27T17:16:24Z,147836707444603,anooptu@gmail.com,CREATE OR REPLACE TABLE AS SELECT,"Map(partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, properties -> {""delta.parquet.compression.codec"":""zstd"",""delta.enableDeletionVectors"":""true""}, statsOnLoad -> false)",null,List(3175228607985161),0826-063441-ydade36q,0,WriteSerializable,false,"Map(numFiles -> 1, numRemovedFiles -> 0, numRemovedBytes -> 0, numOutputRows -> 5, numOutputBytes -> 1393)",null,Databricks-Runtime/16.4.x-scala2.13
0,2026-08-27T17:15:40Z,147836707444603,anooptu@gmail.com,CREATE TABLE,"Map(partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, properties -> {""delta.parquet.compression.codec"":""zstd"",""delta.enableDeletionVectors"":""true""}, statsOnLoad -> false)",null,List(3175228607985161),0826-063441-ydade36q,null,WriteSerializable,true,Map(),null,Databricks-Runtime/16.4.x-scala2.13


In [0]:
%sql
insert into demo values (100, 'John', 1000.0);    
update demo set amount = 3000.0 where id = 100;
delete from demo where id = 1;

num_affected_rows
1


In [0]:
%sql
describe history demo

version,timestamp,userId,userName,operation,operationParameters,job,notebook,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
5,2026-08-27T17:20:01Z,147836707444603,anooptu@gmail.com,DELETE,"Map(predicate -> [""(id#7180L = 1)""])",null,List(3175228607985161),0826-063441-ydade36q,4,WriteSerializable,false,"Map(numRemovedFiles -> 0, numRemovedBytes -> 0, numCopiedRows -> 0, numDeletionVectorsAdded -> 1, numDeletionVectorsRemoved -> 0, numAddedChangeFiles -> 0, executionTimeMs -> 3477, numDeletionVectorsUpdated -> 0, numDeletedRows -> 1, scanTimeMs -> 1740, numAddedFiles -> 0, numAddedBytes -> 0, rewriteTimeMs -> 1736)",null,Databricks-Runtime/16.4.x-scala2.13
4,2026-08-27T17:19:56Z,147836707444603,anooptu@gmail.com,UPDATE,"Map(predicate -> [""(id#6280L = 100)""])",null,List(3175228607985161),0826-063441-ydade36q,3,WriteSerializable,false,"Map(numRemovedFiles -> 1, numRemovedBytes -> 1356, numCopiedRows -> 0, numDeletionVectorsAdded -> 0, numDeletionVectorsRemoved -> 0, numAddedChangeFiles -> 1, executionTimeMs -> 4881, numDeletionVectorsUpdated -> 0, scanTimeMs -> 2858, numAddedFiles -> 1, numUpdatedRows -> 1, numAddedBytes -> 1802, rewriteTimeMs -> 2004)",null,Databricks-Runtime/16.4.x-scala2.13
3,2026-08-27T17:19:49Z,147836707444603,anooptu@gmail.com,WRITE,"Map(mode -> Append, statsOnLoad -> false, partitionBy -> [])",null,List(3175228607985161),0826-063441-ydade36q,2,WriteSerializable,true,"Map(numFiles -> 1, numOutputRows -> 1, numOutputBytes -> 1356)",null,Databricks-Runtime/16.4.x-scala2.13
2,2026-08-27T17:18:09Z,147836707444603,anooptu@gmail.com,SET TBLPROPERTIES,"Map(properties -> {""delta.enableChangeDataFeed"":""true""})",null,List(3175228607985161),0826-063441-ydade36q,1,WriteSerializable,true,Map(),null,Databricks-Runtime/16.4.x-scala2.13
1,2026-08-27T17:16:24Z,147836707444603,anooptu@gmail.com,CREATE OR REPLACE TABLE AS SELECT,"Map(partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, properties -> {""delta.parquet.compression.codec"":""zstd"",""delta.enableDeletionVectors"":""true""}, statsOnLoad -> false)",null,List(3175228607985161),0826-063441-ydade36q,0,WriteSerializable,false,"Map(numFiles -> 1, numRemovedFiles -> 0, numRemovedBytes -> 0, numOutputRows -> 5, numOutputBytes -> 1393)",null,Databricks-Runtime/16.4.x-scala2.13
0,2026-08-27T17:15:40Z,147836707444603,anooptu@gmail.com,CREATE TABLE,"Map(partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, properties -> {""delta.parquet.compression.codec"":""zstd"",""delta.enableDeletionVectors"":""true""}, statsOnLoad -> false)",null,List(3175228607985161),0826-063441-ydade36q,null,WriteSerializable,true,Map(),null,Databricks-Runtime/16.4.x-scala2.13


In [0]:
df1 = spark.read.table('demo')
display(df1)

id,name,amount
100,John,3000.0
2,b,200.0
3,c,300.0
4,d,400.0
5,e,500.0


Fetching CDF

In [0]:
%sql
select * from table_changes('demo',2) -- all changes after version 2

id,name,amount,_change_type,_commit_version,_commit_timestamp
100,John,1000.0,update_preimage,4,2026-08-27T17:19:56Z
100,John,3000.0,update_postimage,4,2026-08-27T17:19:56Z
100,John,1000.0,insert,3,2026-08-27T17:19:49Z
1,a,100.0,delete,5,2026-08-27T17:20:01Z


In [0]:
%sql
delete from demo where id = 2;

num_affected_rows
1


In [0]:
%sql
drop table if exists demo_deleted_data;
create table if not exists demo_deleted_data
(id int,name string , amount double,_change_type string, _commit_version long, _commit_timestamp timestamp
)using delta;

In [0]:
%sql
insert into demo_deleted_data (
select * from table_changes('demo',2) where _change_type = 'delete' ) -- all deletes after version 2

num_affected_rows,num_inserted_rows
3,3


In [0]:
%sql
select * from demo_deleted_data;

id,name,amount,_change_type,_commit_version,_commit_timestamp
3,c,300.0,delete,8,2026-08-27T17:38:42Z
1,a,100.0,delete,5,2026-08-27T17:20:01Z
2,b,200.0,delete,6,2026-08-27T17:23:36Z


Streaming data CDF

In [0]:
from pyspark.sql.functions import col
( spark.readStream.format("delta")
.option('readChangeFeed', 'true')
.option('startingVersion', 2)
.table('demo')
.filter(col('_change_type') == 'delete')
.select('id','name')
.writeStream
.outputMode('append')
.option('checkpointLocation', '/Volumes/useastws/default/deltavolume/checkpoint/')
.trigger(availableNow = True)
.table('demo_deleted_data_streaming'))

'Available now' trigger, will run once, and extract all available data from last check point

In [0]:
%sql
select * from demo_deleted_data_streaming

id,name
1,a
2,b


In [0]:
%sql
delete from demo where id = 3;

num_affected_rows
1


Re running batch CDF query will insert duplicate records to target table, as it doesnt know it has already written some

In [0]:
%sql
insert into demo_deleted_data (
select * from table_changes('demo',2) where _change_type = 'delete' ) -- all deletes after version 2

num_affected_rows,num_inserted_rows
3,3


In [0]:
%sql
select * from demo_deleted_data

id,name,amount,_change_type,_commit_version,_commit_timestamp
3,c,300.0,delete,8,2026-08-27T17:38:42Z
1,a,100.0,delete,5,2026-08-27T17:20:01Z
2,b,200.0,delete,6,2026-08-27T17:23:36Z
3,c,300.0,delete,8,2026-08-27T17:38:42Z
1,a,100.0,delete,5,2026-08-27T17:20:01Z
2,b,200.0,delete,6,2026-08-27T17:23:36Z


But streaming CDF will have an inbuilt checkpointing feature, it wont create duplicates

In [0]:
from pyspark.sql.functions import col
( spark.readStream.format("delta")
.option('readChangeFeed', 'true')
.option('startingVersion', 2)
.table('demo')
.filter(col('_change_type') == 'delete')
.select('id','name')
.writeStream
.outputMode('append')
.option('checkpointLocation', '/Volumes/useastws/default/deltavolume/checkpoint/')
.trigger(availableNow = True)
.table('demo_deleted_data_streaming'))

In [0]:
%sql
select * from demo_deleted_data_streaming

id,name
1,a
2,b
3,c


Automatic streaming of CDF every 2 seconds

In [0]:
from pyspark.sql.functions import col
( spark.readStream.format("delta")
.option('readChangeFeed', 'true')
.option('startingVersion', 2)
.table('demo')
.filter(col('_change_type') == 'delete')
.select('id','name')
.writeStream
.outputMode('append')
.option('checkpointLocation', '/Volumes/useastws/default/deltavolume/checkpoint1/')
.trigger(processingTime = "2 seconds")
.table('demo_deleted_data_streaming_auto'))

.trigger(processingTime = "2 seconds")  -> this will execute the read every 2 seconds continuously until stopped

In [0]:
%sql
select * from demo_deleted_data_streaming_auto

id,name
3,c
1,a
2,b


In [0]:
%sql
delete from demo where id = 4;

num_affected_rows
1


In [0]:
%sql
select * from demo_deleted_data_streaming_auto

id,name
3,c
1,a
2,b
4,d
